# Smart Lender — Exploratory Data Analysis & Model Comparison

This notebook walks through the same pipeline as `train_model.py`, but with
visualizations and commentary at each step. Use it to explore the data,
sanity-check the modeling choices, and inspect why XGBoost (or whichever
model wins on your machine) was selected as the production model.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    from sklearn.ensemble import GradientBoostingClassifier
    print('xgboost not installed -- falling back to GradientBoostingClassifier for this notebook run.')

sns.set_style('whitegrid')
RANDOM_STATE = 42

## 1. Load the data

In [ ]:
df = pd.read_csv('../data/train.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

## 2. Target distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
df['Loan_Status'].value_counts().plot(kind='bar', color=['#0F3D2E', '#9B3B33'], ax=ax)
ax.set_title('Loan_Status distribution')
ax.set_xlabel('Loan Status')
ax.set_ylabel('Count')
plt.show()

df['Loan_Status'].value_counts(normalize=True) * 100

About **69% Y / 31% N** — a moderate class imbalance worth keeping in mind when
reading accuracy numbers; precision/recall per class (below) tells a fuller story.

## 3. Univariate exploration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df['ApplicantIncome'], bins=40, ax=axes[0], color='#0F3D2E')
axes[0].set_title('Applicant Income')
sns.histplot(df['LoanAmount'].dropna(), bins=40, ax=axes[1], color='#B8924A')
axes[1].set_title('Loan Amount')
plt.tight_layout()
plt.show()

Both `ApplicantIncome` and `LoanAmount` are heavily right-skewed with a few
very large outliers — this is exactly why `train_model.py` log-transforms
them (`LoanAmount_log`, `TotalIncome_log`) before feeding them to the models.

## 4. Loan_Status vs. Credit_History

A quick sanity check: credit history is famously the single strongest
predictor in this dataset.

In [ ]:
pd.crosstab(df['Credit_History'], df['Loan_Status'], normalize='index') * 100

Applicants who meet credit guidelines (`Credit_History == 1`) get approved
well over 5x as often as those who don't -- this single field carries most
of the model's predictive power.

## 5. Cleaning, feature engineering & encoding

(mirrors `train_model.py` exactly, so the notebook and the production script stay in sync)

In [ ]:
data = df.copy()
data['Dependents'] = data['Dependents'].replace('3+', '3')

for col in ['Gender', 'Married', 'Dependents', 'Self_Employed']:
    data[col] = data[col].fillna(data[col].mode()[0])

data['LoanAmount'] = data['LoanAmount'].fillna(data['LoanAmount'].median())
data['Loan_Amount_Term'] = data['Loan_Amount_Term'].fillna(data['Loan_Amount_Term'].mode()[0])
data['Credit_History'] = data['Credit_History'].fillna(data['Credit_History'].mode()[0])
data['Dependents'] = data['Dependents'].astype(int)

data['TotalIncome'] = data['ApplicantIncome'] + data['CoapplicantIncome']
data['LoanAmount_log'] = np.log1p(data['LoanAmount'])
data['TotalIncome_log'] = np.log1p(data['TotalIncome'])

data.isnull().sum().sum()  # should be 0

In [ ]:
CATEGORICAL_COLS = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']
FEATURE_COLUMNS = [
    'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
    'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term',
    'Credit_History', 'Property_Area', 'TotalIncome', 'LoanAmount_log', 'TotalIncome_log',
]

encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    encoders[col] = le

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(data['Loan_Status'])
X = data[FEATURE_COLUMNS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train.shape, X_test.shape

## 6. Train all four models

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=9),
}

if XGBOOST_AVAILABLE:
    models['XGBoost'] = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric='logloss', random_state=RANDOM_STATE,
    )
else:
    models['XGBoost (fallback)'] = GradientBoostingClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.08, random_state=RANDOM_STATE,
    )

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    results[name] = {'model': model, 'train_accuracy': train_acc, 'test_accuracy': test_acc}
    print(f'{name:25s}  train={train_acc*100:5.2f}%   test={test_acc*100:5.2f}%')

## 7. Compare model accuracy

In [ ]:
comparison_df = pd.DataFrame({
    name: {'Train Accuracy': r['train_accuracy'] * 100, 'Test Accuracy': r['test_accuracy'] * 100}
    for name, r in results.items()
}).T

comparison_df.plot(kind='bar', figsize=(9, 5), color=['#0F3D2E', '#B8924A'])
plt.title('Model comparison: train vs. test accuracy')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)
plt.xticks(rotation=20)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

comparison_df.round(2)

## 8. Confusion matrix & classification report for the best model

In [ ]:
best_name = max(results, key=lambda n: results[n]['test_accuracy'])
best_model = results[best_name]['model']
print('Best model:', best_name)

y_pred = best_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_encoder.classes_)
disp.plot(cmap='Greens')
plt.title(f'Confusion Matrix — {best_name}')
plt.show()

## 9. Feature importance (tree-based models)

If the selected model exposes `feature_importances_` (Decision Tree, Random
Forest, or XGBoost), this shows which inputs drive the prediction most.

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=FEATURE_COLUMNS)
    importances = importances.sort_values(ascending=True)
    importances.plot(kind='barh', figsize=(7, 6), color='#0F3D2E')
    plt.title(f'Feature importance — {best_name}')
    plt.tight_layout()
    plt.show()
else:
    print(f'{best_name} does not expose feature_importances_ (e.g. KNN).')

## 10. Conclusion

`train_model.py` runs this exact pipeline non-interactively and persists the
winning model + preprocessing objects to `model/` for the Flask app
(`app.py`) to load at request time. Re-run that script any time the dataset
changes.